# Tethys-Chloris (T&C): spatially-distributed modeling

## Material
* Case Study Materials
  * Script to create the model inputs (`create_data_2d.jl`)

* Input Data
  * Files containing CO2 air concentration
  * Digital elevation model and catchment geodata (`GeoExample.mat`)
  * Meteorological station data from the first exercise (`Data_CH-Cha_run.mat`)

* Visualization Tools
  * Functions for results visualizations (Plots.jl)

## Objectives

This session is designed to help students learn how to prepare input files and execute a basic distributed simulation using the spatially-distributed (2D) version of the T&C model. It extends the plot-scale workflows of the previous exercises to a small catchment, where terrain, flow routing and spatially varying inputs become relevant.

The tutorial here will focus on one virtual catchment located at [Chamau, ZG](https://www.google.com/maps/place/47%C2%B012'36.8%22N+8%C2%B024'38.3%22E/@47.2102278,8.4099778,273m/data=!3m2!1e3!4b1!4m4!3m3!8m2!3d47.2102269!4d8.4106451?entry=ttu), using the same meteorological data as in the first exercise.

# Part A: input preparation, model setup and simulation

## 1. Prepare the digital elevation input file

In addition to the meteorological input data prepared in the previous exercise, the spatial (2D) application needs topographic information for every computation cell: elevation, slope, aspect, flow directions, the stream network, land cover and soil texture.

All this information is bundled in the file **`data/GeoExample.mat`**. Load it and inspect its contents:

In [ ]:
using MAT: matread

geo = matread(joinpath(@__DIR__, "data", "GeoExample.mat"))
println(sort(collect(keys(geo))))

Reflect on the study site and the catchment characteristics:

- *What is the size of the topography? How many computation cells does it have?*
- *What is the spatial resolution?*
- *How many types of vegetation are there? (check the variable `ksv`, remembering that in the demo the codes 1–4 are collapsed to two classes before the simulation)*
- *What are the soil properties in the domain? (check `PSAN`, `PCLA`, `PORG`)*

## 2. Understand the case study

As for the 1D application, understanding the case study is essential before running simulations. The demo parameter file distinguishes two land cover types through the variable `ANSWER` (in the MATLAB version of the demo), which is loaded from the digital elevation file:

- If `ANSWER = 1`, the cell is 100% covered by vegetation, consisting only of **high vegetation** (wood). The root depth of the high vegetation is set to 400 mm.
- If `ANSWER = 2`, the cell is also 100% covered by canopy but consists only of **low vegetation** (grass/shrub). The root depth of the low vegetation is set to 250 mm.

This parameterization enables the model to differentiate vegetation properties based on the land cover type. In the Julia setup the same information is stored in the generated parameter file (`data/exercise_2d_parameters.yaml`), with one block per land cover class.

Before running the simulation, answer the following questions:

- *What are the simulation period and the time step?*
- *What temporal patterns do you expect in Leaf Area Index (LAI), evapotranspiration and soil moisture during this period?*
- *Where do you expect the wettest and driest cells of the catchment to be, and why?*

## 3. Understand the structure of the model

The 2D application shares most of its physics with the plot-scale model you already ran in the previous exercises (per-cell SVAT, vegetation dynamics, soil thermal and hydrologic processes). This section therefore focuses on the elements that are **new in the distributed case**:

- **Flow routing (`T_flow`).** The file `GeoExample.mat` contains a routing matrix that connects every cell to its downstream neighbour(s). After all per-cell simulations complete at a time step, the routing module moves surface and subsurface water along these connections towards the catchment outlet. This lateral redistribution is what the plot-scale model cannot represent.

- **Spatially distributed forcing (lapse rates, `grad`).** The meteorological station measures at one elevation. To obtain forcing fields over the catchment, temperature, wind, humidity and radiation are extrapolated with elevation lapse rates (e.g. `grad1 = -0.0045 °C/m` for temperature in `GO_TeC_Demo.m`).

- **Terrain radiation (`SvF`, `Ct`, `ShF`).** Sky view factors, terrain configuration factors and shadow effects modify the shortwave radiation received by each cell depending on slope, aspect and surrounding topography.

- **Spatial snapshots (`tstore`).** The time steps at which spatially-averaged maps are saved. In the MATLAB demo `tstore = [40, 2160, 4320, 6480, 8640]`; in the Julia setup the equivalent option is `tstore`, passed to `run_simulation`.

- **Output groups.** The MATLAB output manager writes `AVG`/`STD` (catchment-average time series and their standard deviation), `PIXEL` (time series at tracked points) and `SPATIAL` (maps at the `tstore` steps) files. The Julia run streams the same content into a single NetCDF with the groups `spatial_mean`, `spatial_std`, `point_1`, `time_average` and `grid`.

Open the script **`create_data_2d.jl`** and identify where each of these elements is created or configured. Then answer:

- *What does `T_flow` control that elevation and slope alone do not?*
- *Why must meteorological forcing be extrapolated with lapse rates, and which variables are most sensitive to this extrapolation?*
- *What would change in the results if the outlet cell (`Xout`, `Yout`) were moved to a different corner of the catchment?*

## 4. Ready to run the simulation

The script **`create_data_2d.jl`** converts the raw MATLAB files into the two inputs the Julia model needs: a NetCDF file (forcing + grid) and a YAML file (parameters for both land cover classes). Before running the model, verify that the parameters are set as intended — in particular that the bedrock conductivity `Kbot` in the `rock:` block of the YAML is filled in (set it to `0.11` mm/h, the value used in the MATLAB demo, in **both** land cover blocks).

Generate the inputs (this takes a few seconds):

In [ ]:
include(joinpath(@__DIR__, "create_data_2d.jl"))

Follow the same steps as in exercise 1 (section 6) to initialize the model, this time with the 2D model type:

In [ ]:
using NCDatasets
using YAML
using Roots
using TethysChloris

FT = Float64
data_dir = joinpath(@__DIR__, "data")

netcdfpath = joinpath(data_dir, "exercise_2d_data.nc")
yamlpath = joinpath(data_dir, "exercise_2d_parameters.yaml")

model, forcing = initialize_model(Model2D{FT}, netcdfpath, yamlpath);

In [ ]:
options = ModelOptions(
    SoilBiogeochemistry = false,
    SoilTemperature = true,
    FreezingSoil = true,
    VegetationSnowInteractions = true,
    OPT_CR = RootsNonBracketingStrategy(FT),
    OPT_ST = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_ST2 = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_VD = ODEOptions(; abstol = 0.05),
);

During the exercise session, run the simulation for a short period first to validate the setup. Ten timesteps take well under a minute (the equivalent MATLAB check takes about half an hour on a laptop):

In [ ]:
diagnostics, manager = run_simulation(
    model,
    forcing;
    options = options,
    run_options = SpatialRunOptions(
        NN = 10,
        output_level = AllOutputs(),
        output_filename = joinpath(data_dir, "smoke_results.nc"),
        compress = true,
        parallel = false,
        tstore = [10],
    ),
);

The full window is 1 May to 3 October 2005 (3743 hourly steps), and takes several hours on a laptop (the MATLAB original used an 18-worker parallel pool for the same task). The result streams to `exercise_2d_results.nc`. To enable the cell, uncomment its body:

In [ ]:
# FULL RUN: several hours on a laptop. Uncomment the body to enable.
error("Uncomment the body of this cell to run the full simulation")

# diagnostics, manager = run_simulation(
#     model,
#     forcing;
#     options = options,
#     run_options = SpatialRunOptions(
#         output_level = AllOutputs(),
#         output_filename = joinpath(data_dir, "exercise_2d_results.nc"),
#         compress = true,
#         parallel = false,
#         tstore = [40, 2160],
#     ),
# );

## 5. Checking the results

To quickly inspect the simulation results, read the output NetCDF and reproduce the kind of plots the MATLAB scripts `GRAPH_TEC` (spatial maps) and `GRAPH_TS` (time series of spatially-averaged variables) produce. These quick checks allow you to verify the trends, patterns, and spatial distributions of key variables like temperature, soil moisture, latent heat and leaf area index.

The output file mirrors the MATLAB demo's outputs:

| Group | Content | MATLAB analogue |
|---|---|---|
| `spatial_mean` / `spatial_std` | catchment-average time series and their standard deviation | `OUTPUT_..._AVG.dat`, `OUTPUT_..._STD.dat` |
| `point_1` | time series at the tracked grid point | `OUTPUT_..._PIXEL_1.dat` |
| `time_average` | spatial maps of time-averaged state at the `tstore` steps | `OUTPUT_..._SPATIAL_t.mat` |
| `grid` | static grid data (DTM, land use, ...) | `GeoExample.mat` |

In [ ]:
results = NCDataset(joinpath(data_dir, "exercise_2d_results.nc"))
sm = results.group["spatial_mean"]
ta = results.group["time_average"]
pt = results.group["point_1"]
nothing

### 5.1 Time series of spatially-averaged variables (GRAPH_TS)

In [ ]:
using Plots

days = collect(0:length(Array(sm["Ts"])) - 1) ./ 24

p1 = plot(days, Array(sm["Ts"]), label = "Ts", ylabel = "Surface temperature [°C]", legend = :best)
p2 = plot(days, Array(sm["QE"]), label = "QE", ylabel = "Latent heat [W/m²]", legend = :best)
p3 = plot(days, Array(sm["H"]), label = "H", ylabel = "Sensible heat [W/m²]", legend = :best)
p4 = plot(days, Array(sm["ZWT"]), label = "ZWT", ylabel = "Water table depth [mm]", legend = :best)
plot(p1, p2, p3, p4, layout = (4, 1), size = (800, 900), xlabel = "Time [days]")

Compare your plots with Figure 9 of the handout (example results from the 5-month simulation). Then answer:

- *Relate the evaporation peaks in `QE` to rainfall events and to the seasonal trajectory of `Ts`. Which limitation dominates in early May, which in late summer?*
- *Does the water table depth (`ZWT`) rise during the May rains in your run? What does this tell you about the balance between infiltration and bedrock drainage?*

### 5.2 Spatial maps (GRAPH_TEC)

In [ ]:
Ts_maps = Array(ta["Ts"]) # (m_cell, n_cell, snapshots)
snapshots = ["t = 40 (spin-up)", "t = 2160 (~3 months)", "t = end"]

plots = [heatmap(Ts_maps[:, :, i]'; title = "Ts, $(snapshots[i])", clims = (0, 30), aspect_ratio = 1)
         for i in 1:size(Ts_maps, 3)]
plot(plots...; layout = (1, length(plots)), size = (300 * length(plots), 300))

In [ ]:
V_maps = Array(ta["V"]) # (m_cell, n_cell, layers, snapshots)
layer = 1

plots = [heatmap(V_maps[:, :, layer, i]'; title = "V (layer $layer), $(snapshots[i])", aspect_ratio = 1)
         for i in 1:size(V_maps, 4)]
plot(plots...; layout = (1, length(plots)), size = (300 * length(plots), 300))

- *The maps inherit the catchment's topography. Where does the soil stay wettest, and how does the elevation gradient and the flow routing explain it?*
- *How does the pattern change between the spin-up snapshot (t = 40) and the end of the simulation?*

### 5.3 Soil moisture profile at the tracked point

In [ ]:
V = Array(pt["V"]) # (time, layers)

heatmap(days, 1:size(V, 2), V'; xlabel = "Time [days]", ylabel = "Soil layer",
    colorbar_title = "Water content [‰]", title = "Soil moisture profile at point 1")

- *During which periods does the profile dry from the top down, and when is it rewetted?*
- *Try plotting a different layer or a different variable (e.g. `Tdp`, the soil temperature profile) at the tracked point.*

In [ ]:
close(results)

### Find out more

[Laboratory of Catchment Hydrology and Geomorphology](https://www.epfl.ch/labs/change/)
[T&C source code (Julia)](github.com/CHANGE-EPFL/TethysChloris.jl)
[T&C source code (MATLAB)](https://github.com/simonefatichi/TeC_Source_Code)


### References 

Fatichi, S., Ivanov, V. Y., & Caporali, E. (2012). A mechanistic ecohydrological model to investigate complex interactions in cold and warm water‐controlled environments: 1. Theoretical framework and plot‐scale analysis. *Journal of Advances in Modeling Earth Systems, 4*(2).

Remondi, F., Botter, M., Burlando, P., & Fatichi, S. (2019). Variability of transit time distributions with simple catchments: the role of the spatial distribution of rainfall. *Hydrology and Earth System Sciences, 23*(8).